<a href="https://colab.research.google.com/github/ChirilaTeodora/Electronic-Component-Classifier/blob/main/Antrenare_Finala_Componente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shutil
import json
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# ==========================================
# 1. CONFIGURATION
# ==========================================
# ==========================================
# 1. CONFIGURATION
# ==========================================
# Look for either zip or rar
ZIP_FILE = 'Componente_Reale.zip'
RAR_FILE = 'Componente_Reale.rar'
BASE_DIR = '/content/dataset'
IMG_SIZE = 128
BATCH_SIZE = 32

# Clean up previous runs
if os.path.exists(BASE_DIR): shutil.rmtree(BASE_DIR)

print(">>> PREPARING DATASET...")

# Check what file exists
if os.path.exists(ZIP_FILE):
    print(f"File found: {ZIP_FILE}. Unzipping...")
    os.system(f'unzip -q {ZIP_FILE} -d {BASE_DIR}')
elif os.path.exists(RAR_FILE):
    print(f"File found: {RAR_FILE}. Installing unrar and extracting...")
    os.system('apt-get install unrar')
    os.makedirs(BASE_DIR, exist_ok=True)
    os.system(f'unrar x {RAR_FILE} {BASE_DIR}')
else:
    print(f"!!! ERROR: No dataset found. Please upload '{ZIP_FILE}' or '{RAR_FILE}' to Colab.")

# Locate the actual data folder (handles cases where zip/rar contains a parent folder)
possible_path = os.path.join(BASE_DIR, 'Componente_Reale')
PATH_TO_DATA = possible_path if os.path.exists(possible_path) else BASE_DIR
print(f"Data directory: {PATH_TO_DATA}")

# ==========================================
# 2. DATA GENERATORS & AUGMENTATION
# ==========================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)
val_datagen = ImageDataGenerator(rescale=1./255)

try:
    train_generator = train_datagen.flow_from_directory(
        os.path.join(PATH_TO_DATA, 'train'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )

    validation_generator = val_datagen.flow_from_directory(
        os.path.join(PATH_TO_DATA, 'validation'),
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical'
    )
except Exception as e:
    print(f"Error initializing data generators: {e}")
    train_generator = None
    validation_generator = None

# ==========================================
# 3. MODEL BUILD (Transfer Learning)
# ==========================================
if train_generator:
    print(f"Detected Classes: {list(train_generator.class_indices.keys())}")

    # Load MobileNetV2 without top layers
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = False # Freeze base layers

    # Add custom head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(len(train_generator.class_indices), activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # ==========================================
    # 4. TRAINING WITH EARLY STOPPING
    # ==========================================
    # EarlyStopping: Stops training if validation loss doesn't improve for 'patience' epochs.
    # restore_best_weights: Ensures we save the best model, not the last one.
    early_stopping = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )

    print(">>> STARTING TRAINING...")
    history = model.fit(
        train_generator,
        epochs=30,  # Max epochs (will stop earlier if needed)
        validation_data=validation_generator,
        callbacks=[early_stopping]
    )

    # ==========================================
    # 5. SAVING ARTIFACTS
    # ==========================================
    # Note: Removed space from filenames for consistency
    MODEL_FILENAME = 'component_classifier_FINAL.h5'
    CLASSES_FILENAME = 'component_classes_FINAL.json'

    model.save(MODEL_FILENAME)

    # Save class mapping
    class_indices = {v: k for k, v in train_generator.class_indices.items()}
    with open(CLASSES_FILENAME, 'w') as f:
        json.dump(class_indices, f)

    print(">>> SAVED MODEL & CLASSES.")

    # Trigger download in Colab
    try:
        from google.colab import files
        files.download(MODEL_FILENAME)
        files.download(CLASSES_FILENAME)
    except ImportError:
        print("Not in Colab: Files saved locally but not downloaded.")


>>> PREPARING DATASET...
File found: Componente_Reale.rar. Installing unrar and extracting...
Data directory: /content/dataset/Componente_Reale
Found 455 images belonging to 4 classes.
Found 65 images belonging to 4 classes.
Detected Classes: ['Capacitor', 'Inductor', 'Resistor', 'Transistor']
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
>>> STARTING TRAINING...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 24s 1s/step - accuracy: 0.5374 - loss: 1.1353 - val_accuracy: 0.8615 - val_loss: 0.4332
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 15s 951ms/step - accuracy: 0.9203 - loss: 0.2210 - val_accuracy: 0.8769 - val_loss: 0.3805
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 904ms/step - accuracy: 0.9629 - loss: 0.1327 - val_accuracy: 0.8923 - val_loss: 0.3127
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 992ms/step - accuracy: 0.9559 - loss: 0.1122 - val_accuracy: 0.8769 - val_loss: 0.2879
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 937ms/step - accuracy: 0.9729 - loss: 0.0863 - val_accuracy: 0.8769 - val_loss: 0.2896
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 903ms/step - accuracy: 0.9770 - loss: 0.0661 - val_accuracy: 0.9077 - val_loss: 0.3255
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 870ms/step - accuracy: 0.9829 - loss: 0.0397 - val_accuracy: 0.9231 - val_loss: 0.2717
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 881ms/step - accuracy: 0.9935 - loss: 0.0327 - val_accurac

>>> SAVED MODEL & CLASSES.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>